In [1]:
# integrated LLM for ---> converting structured tissue labels ===> controlled natural language explanations;

In [2]:
import pandas as pd

profile_df = pd.read_csv(r"E:\404_Found\H&E Stained Histological Nucleus Instancve Segmentation by YOLOv8x\PanNuke_Results\Dump_Seg_Results\tissue_profiles.csv")
profile_df.head()

,image_name,density_label,size_label,tissue_profile
0,1-100.png,Medium Density,Medium Nuclei,Moderate Tissue Structure
1,1-1000.png,Low Density,Small Nuclei,Sparse & Uniform Tissue
2,1-1009.png,Medium Density,Medium Nuclei,Moderate Tissue Structure
3,1-1012.png,Low Density,Medium Nuclei,Moderate Tissue Structure
4,1-1020.png,High Density,Small Nuclei,Crowded & Small Cells


In [3]:
# generating the prompt : Prompt Engineering;

def build_prompt(row):
    return f"""
You are a biomedical image analysis assistant.

Given the following tissue characteristics:

Density: {row['density_label']}
Nucleus Size: {row['size_label']}
Tissue Profile: {row['tissue_profile']}

Task:
Write a short, clear, and professional explanation of the tissue structure.

Rules:
- Do NOT invent medical claims
- Do NOT mention diseases unless explicitly implied
- Keep it 1–2 sentences
- Focus only on structural interpretation
"""

In [4]:
import ollama

# defining the LLM call : 

def get_explanation(prompt):
    response = ollama.chat(
        model="gemma4:e4b",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    return response['message']['content']

In [ ]:
# running the LLM(gemma4:e4b) on dataset, but for testing running on only top 5 rows first;
# this is for testing as i am not saving this explantion(response from the llm) to the CSV;

test_rows = profile_df.head(5)

for _, row in test_rows.iterrows():
    prompt = build_prompt(row)
    print(get_explanation(prompt))
    print("-" * 50)

The tissue exhibits a moderate overall structure, maintaining a medium density profile. The structural assessment is supported by the presence of medium-sized nuclei throughout the sample.
--------------------------------------------------
The tissue exhibits a sparse and uniform structural pattern characterized by a low overall density. This architecture is further defined by the presence of small nuclei, indicating low cellular packing.
--------------------------------------------------
The tissue exhibits a moderate overall structure, characterized by medium density and uniformly sized nuclei. This suggests a balanced and typical cellular arrangement within the analyzed region.
--------------------------------------------------
The tissue exhibits a low-density appearance with average-sized nuclei, indicating a relatively open or sparsely packed cellular arrangement. Overall, the structure is moderate, suggesting discernible tissue organization.
-------------------------------------

In [7]:
# now i am gonna save the EXPLANATION(RESPONSE FROM LLM) into the CSV, but as i have ~2k images/rows, i will generate and save for the 1st 5 images and save, for explanation of other images ---> RUN U THE CODE IN UR GPU!!!!


profile_df["llm_explanation"] = "LLM skipped this one. Even GPUs deserve boundaries"

for i in range(min(5, len(profile_df))):
    row = profile_df.iloc[i] # .iloc[i] → gets row
    
    prompt = build_prompt(row)
    
    try:
        explanation = get_explanation(prompt)
    except:
        explanation = "LLM error"
    
    profile_df.at[i, "llm_explanation"] = explanation # .at[i, col] ---> updates that row

In [11]:
# saving that CSV with EXPLANATIONS(FROM LLM)

profile_df.to_csv(r"E:\404_Found\H&E Stained Histological Nucleus Instancve Segmentation by YOLOv8x\PanNuke_Results\Dump_Seg_Results\tissue_profiles_with_explanations.csv", index=False)